In [1]:
!pip install -q transformers datasets accelerate bitsandbytes peft sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Load Dataset

In [2]:
from datasets import load_dataset

In [3]:
# Load dataset from Hugging Face
dataset = load_dataset("Novaspree/5W_Factify_QA")

# Check dataset structure
print(dataset)
print(dataset["train"][0])  # Print the first sample

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

processed_claims_with_qa.json:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/391041 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['claim', 'evidence', '5W_aspects', 'qa_pairs'],
        num_rows: 391041
    })
})
{'claim': 'China’s famed wandering elephants are on the move again, heading southwest while a male who broke from the herd is still keeping his distance. https://t.co/o5j7PDDveJ', 'evidence': "By Julia Hollingsworth and Zixu Wang, CNNUpdated 1:03 AM ET, Fri June 11, 2021  (CNN)At least a dozen buzzing drones monitor them around the clock.  Wherever they go, they're escorted by police.  And when they eat or sleep, they're watched by millions online.  CNN's Jessie Yeung contributed to this report.  ", '5W_aspects': {'what': [], 'when': [], 'where': ['China'], 'who': [], 'why': None}, 'qa_pairs': [{'answer': 'China', 'question': 'Where did the event take place?'}]}


# Prepare Dataset

In [4]:
from transformers import AutoTokenizer

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [5]:
# Load tokenizer
model_name = "microsoft/phi-1_5"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token (since Phi-1.5 has no default pad token)
tokenizer.pad_token = tokenizer.eos_token  # ✅ Fix padding issue


tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [8]:
def tokenize_function(batch):
    texts = [claim + " " + evidence if claim and evidence else "" for claim, evidence in zip(batch["claim"], batch["evidence"])]

    encoding = tokenizer(
        texts,  # ✅ Now passing a list of strings, not a single list
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    encoding["labels"] = encoding["input_ids"].clone()  # ✅ Ensure labels match input_ids
    return encoding


In [9]:
# Apply tokenization efficiently
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Convert to PyTorch format
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/391041 [00:00<?, ? examples/s]

# Fine Tuning

In [17]:
!pip install -q peft

In [18]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from transformers import TrainingArguments
from transformers import Trainer
from peft import LoraConfig, get_peft_model

In [19]:
# Enable 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="bfloat16",
    bnb_4bit_use_double_quant=True,
)


In [20]:
# Load Phi-1.5 with quantization
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-1_5",
    quantization_config=bnb_config,
    device_map="auto"
)

# ✅ Attach LoRA adapters for training
lora_config = LoraConfig(
    r=8,  # Low-rank dimension
    lora_alpha=16,  # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Apply LoRA to key attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # ✅ Check trainable parameter count

print("✅ LoRA Adapters Attached & Model Ready for Fine-Tuning")

trainable params: 1,572,864 || all params: 1,419,843,584 || trainable%: 0.1108
✅ LoRA Adapters Attached & Model Ready for Fine-Tuning


In [21]:
training_args = TrainingArguments(
    output_dir="./phi_finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",  # ✅ Disable WandB logging
    run_name="ssu_phi1.5"  # Optional: Set a custom run name
)


In [23]:
# Select the 'train' split
train_dataset = tokenized_dataset["train"]  # ✅ Fix: Select the correct dataset split

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset  # ✅ Now using the correct split
)

# Start training
trainer.train()

# Save fine-tuned model
model.save_pretrained("phi_finetuned")

print("✅ Fine-Tuning Complete & Model Saved")


Step,Training Loss
10,4.491300
20,4.340300
30,4.355200
40,3.933100
50,3.881700
60,3.696800
70,3.543800
80,3.394800
90,2.861800
100,2.804200


Step,Training Loss
10,4.491300
20,4.340300
30,4.355200
40,3.933100
50,3.881700
60,3.696800
70,3.543800
80,3.394800
90,2.861800
100,2.804200


✅ Fine-Tuning Complete & Model Saved


# Task Vectors

In [26]:
import torch
import copy
from transformers import Trainer

In [31]:
import torch
import copy
from transformers import Trainer

# Function to introduce Random Labeling Loss
def random_labeling_loss(example):
    if "labels" in example:
        labels_tensor = torch.tensor(example["labels"]).clone().detach()
        example["labels"] = torch.roll(labels_tensor, shifts=1).tolist()
    return example

# Function to apply SSU Unlearning on a Smaller Dataset
def apply_ssu_unlearning(model, aspect, dataset):
    print(f"🔹 Applying SSU Unlearning for {aspect}...")

    # ✅ Select the 'train' split explicitly
    dataset = dataset["train"]

    # ✅ Reduce dataset size to 10% for faster processing
    subset_size = int(0.01 * len(dataset))  # 10% of total samples
    aspect_data = dataset.filter(lambda x: aspect in x["5W_aspects"]).select(range(subset_size))

    # ✅ Apply Random Labeling Loss
    aspect_data = aspect_data.map(random_labeling_loss)

    # ✅ Create a copy of the fine-tuned model
    model_unlearned = copy.deepcopy(model)

    # Define training arguments for unlearning
    training_args_unlearn = TrainingArguments(
        output_dir=f"./model_unlearned_{aspect}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,  # ✅ 1 Epoch to prevent excessive runtime
        learning_rate=2e-5,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none"
    )

    # Train the model to unlearn
    trainer_unlearn = Trainer(
        model=model_unlearned,
        args=training_args_unlearn,
        train_dataset=aspect_data
    )

    trainer_unlearn.train()

    # Save the unlearned model
    model_unlearned.save_pretrained(f"model_unlearned_{aspect}")

    print(f"✅ SSU Unlearning Complete for {aspect}")

# ✅ Apply SSU for each "W" aspect on a reduced dataset
for w in ["who", "what", "when", "where", "why"]:
    apply_ssu_unlearning(model, w, tokenized_dataset)

print("✅ All 5W Aspects Partially Unlearned & Models Saved")


🔹 Applying SSU Unlearning for who...


Map:   0%|          | 0/3910 [00:00<?, ? examples/s]

<ipython-input-31-183bf1b2df5b>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels_tensor = torch.tensor(example["labels"]).clone().detach()


Step,Training Loss
10,10.055000
20,9.261600
30,8.463100
40,7.513800
50,7.241500
60,6.209300
70,5.849200
80,5.532400
90,4.941400
100,4.603200


✅ SSU Unlearning Complete for who
🔹 Applying SSU Unlearning for what...


...

Filter:   0%|          | 0/391041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3910 [00:00<?, ? examples/s]

<ipython-input-31-183bf1b2df5b>:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels_tensor = torch.tensor(example["labels"]).clone().detach()


Step,Training Loss
10,10.055100
20,9.262000
30,8.463500
40,7.516100
50,7.245600
60,6.212400
70,5.851500
80,5.534900
90,4.944300
100,4.607200


✅ SSU Unlearning Complete for why
✅ All 5W Aspects Partially Unlearned & Models Saved


In [32]:
import torch
from transformers import AutoModelForCausalLM

# Load the original fine-tuned model (before unlearning)
model_finetuned = AutoModelForCausalLM.from_pretrained("phi_finetuned").to("cpu")

# Function to compute task vectors
def compute_task_vector(model_before, model_after):
    task_vector = {
        name: model_after.state_dict()[name] - model_before.state_dict()[name]
        for name in model_after.state_dict()
        if name in model_before.state_dict()
    }
    return task_vector

# Dictionary to store task vectors for each 5W aspect
task_vectors = {}

# Compute task vectors for each "W"
for w in ["who", "what", "when", "where", "why"]:
    print(f"🔹 Computing task vector for {w}...")

    # Load the unlearned model
    model_unlearned = AutoModelForCausalLM.from_pretrained(f"model_unlearned_{w}").to("cpu")

    # Compute and store the task vector
    task_vectors[w] = compute_task_vector(model_finetuned, model_unlearned)

    # Free up memory
    del model_unlearned
    torch.cuda.empty_cache()

print("✅ Task Vectors Computed for Each 5W Aspect")

🔹 Computing task vector for who...
🔹 Computing task vector for what...
🔹 Computing task vector for when...
🔹 Computing task vector for where...
🔹 Computing task vector for why...
✅ Task Vectors Computed for Each 5W Aspect


In [34]:
import torch
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")
tokenizer.pad_token = tokenizer.eos_token  # ✅ Ensure correct padding

# Function to evaluate forgetting
def evaluate_unlearning(model, tokenizer, test_samples):
    model.eval()
    results = {"Who": 0, "What": 0, "When": 0, "Where": 0, "Why": 0}

    for entry in test_samples:
        input_text = entry["claim"] + " " + entry["evidence"]
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=50)

        prediction = tokenizer.decode(output[0], skip_special_tokens=True)

        for key in results.keys():
            if key.lower() in entry["5W_aspects"]:
                if any(word in prediction for word in entry["5W_aspects"][key.lower()]):
                    results[key] += 1

    return results

# Dictionary to store evaluation results
performance_results = {}

# Evaluate forgetting for each 5W aspect
for w in ["who", "what", "when", "where", "why"]:
    print(f"🔍 Evaluating forgetting for {w}...")

    # Load unlearned model
    model = AutoModelForCausalLM.from_pretrained(f"model_unlearned_{w}").to("cuda")

    # Select a small test set
    test_samples = tokenized_dataset["train"].select(range(10)).to_list()  # ✅ Test on first 10 samples

    # Evaluate forgetting performance
    performance_results[w] = evaluate_unlearning(model, tokenizer, test_samples)

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

print("✅ Forgetting Evaluation Complete")


🔍 Evaluating Forgetting Accuracy for who...
🔍 Evaluating Forgetting Accuracy for what...
🔍 Evaluating Forgetting Accuracy for when...
🔍 Evaluating Forgetting Accuracy for where...
🔍 Evaluating Forgetting Accuracy for why...
✅ Forgetting Accuracy Evaluation Complete
{'who': 22.0, 'what': 94.0, 'when': 66.0, 'where': 46.0, 'why': 0.0}


In [42]:
def tokenize_function(batch):
    # ✅ Replace None values with empty strings
    text_inputs = [" ".join([str(c) if c else "", str(e) if e else ""]) for c, e in zip(batch["claim"], batch["evidence"])]

    encoding = tokenizer(
        text_inputs,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    encoding["labels"] = encoding["input_ids"].copy()  # ✅ Store labels
    encoding["5W_aspects"] = batch["5W_aspects"]  # ✅ Keep 5W aspects

    return encoding

# ✅ Retokenize dataset while keeping '5W_aspects'
tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/391041 [00:00<?, ? examples/s]

In [45]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")
tokenizer.pad_token = tokenizer.eos_token  # ✅ Fix padding issue

# Function to get model responses for QA pairs
def get_model_responses(model, tokenizer, qa_samples):
    model.eval()
    responses = []

    for qa in qa_samples:
        question = qa.get("question", "")  # ✅ Use .get() to avoid KeyError
        expected_answer = qa.get("answer", "")

        inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=50)

        prediction = tokenizer.decode(output[0], skip_special_tokens=True)

        responses.append({
            "Question": question,
            "Expected Answer": expected_answer,
            "Model Response": prediction
        })

    return responses

# Dictionary to store QA responses for each 5W aspect
qa_responses = {}

# Check responses for both forgetting and retaining questions
for w in ["who", "what", "when", "where", "why"]:
    print(f"🔍 Generating QA responses for {w} unlearning...")

    # Load the unlearned model
    model = AutoModelForCausalLM.from_pretrained(f"model_unlearned_{w}").to("cuda")

    # ✅ Extract small sets of QA pairs
    qa_samples_forgetting = [
        qa for entry in tokenized_dataset["train"] if "qa_pairs" in entry and w in entry["5W_aspects"]
        for qa in entry["qa_pairs"]
    ][:3]  # ✅ Only 3 samples for quick testing

    qa_samples_retaining = [
        qa for entry in tokenized_dataset["train"] if "qa_pairs" in entry and w not in entry["5W_aspects"]
        for qa in entry["qa_pairs"]
    ][:3]  # ✅ Only 3 samples for quick testing

    # ✅ Get model responses
    forgetting_responses = get_model_responses(model, tokenizer, qa_samples_forgetting)
    retaining_responses = get_model_responses(model, tokenizer, qa_samples_retaining)

    qa_responses[w] = {
        "Forgetting QA Responses": forgetting_responses,
        "Retaining QA Responses": retaining_responses
    }

    # ✅ Free GPU memory
    del model
    torch.cuda.empty_cache()

print("✅ QA Pair Responses Generated")

# ✅ Print sample responses
print(json.dumps(qa_responses, indent=4))


🔍 Generating QA responses for who unlearning...
🔍 Generating QA responses for what unlearning...
🔍 Generating QA responses for when unlearning...
🔍 Generating QA responses for where unlearning...
🔍 Generating QA responses for why unlearning...
✅ QA Pair Responses Generated
{
    "who": {
        "Forgetting QA Responses": [
            {
                "Question": "Where did the event take place?",
                "Expected Answer": "China",
                "Model Response": "Where did the event take place?."
            },
            {
                "Question": "Who is mentioned in the claim?",
                "Expected Answer": "Narendra Modi",
                "Model Response": "Who is mentioned in the claim?."
            },
            {
                "Question": "Who is mentioned in the claim?",
                "Expected Answer": "Vladimir Putin",
                "Model Response": "Who is mentioned in the claim?."
            }
        ],
        "Retaining QA Responses": []

In [46]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5")
tokenizer.pad_token = tokenizer.eos_token  # ✅ Fix padding issue

# Function to compute retain accuracy
def compute_retain_accuracy(model, tokenizer, retain_samples):
    model.eval()
    successful_retain = 0
    total_cases = len(retain_samples)

    for qa in retain_samples:
        question = qa.get("question", "")
        expected_answer = qa.get("answer", "")

        inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=50)

        prediction = tokenizer.decode(output[0], skip_special_tokens=True)

        # ✅ Check if the model's response matches the expected answer
        if expected_answer in prediction:
            successful_retain += 1  # ✅ Successful retention

    # Compute retain accuracy
    retain_accuracy = (successful_retain / total_cases) * 100 if total_cases > 0 else 0
    return retain_accuracy

# Dictionary to store retain accuracy results
retain_accuracy_results = {}

# Evaluate retain accuracy for each 5W aspect
for w in ["who", "what", "when", "where", "why"]:
    print(f"🔍 Evaluating Retain Accuracy for {w}...")

    # Load the unlearned model
    model = AutoModelForCausalLM.from_pretrained(f"model_unlearned_{w}").to("cuda")

    # ✅ Extract small retain set (3 samples per aspect)
    retain_samples = [
        qa for entry in tokenized_dataset["train"] if "qa_pairs" in entry and w not in entry["5W_aspects"]
        for qa in entry["qa_pairs"]
    ][:3]  # ✅ Only 3 samples for quick testing

    # Compute retain accuracy
    retain_accuracy_results[w] = compute_retain_accuracy(model, tokenizer, retain_samples)

    # ✅ Free GPU memory
    del model
    torch.cuda.empty_cache()

print("✅ Retain Accuracy Evaluation Complete")
print(retain_accuracy_results)


🔍 Evaluating Retain Accuracy for who...
🔍 Evaluating Retain Accuracy for what...
🔍 Evaluating Retain Accuracy for when...
🔍 Evaluating Retain Accuracy for where...
🔍 Evaluating Retain Accuracy for why...
✅ Retain Accuracy Evaluation Complete
{'who': 0, 'what': 0, 'when': 0, 'where': 0, 'why': 0}
